# Spark Optimization

Issues we are trying to solve:
- Shuffle
- Data skew
- Small files

General Guidelines/ Best Practices:
- filter early
- column pruning
- avoid UDFs
- Use AQE 

Optimization techniques (Dataframes)
- repartition vs coalesce
- broadcast joins 
- cache/persist
- salting



## Setup

In [0]:
# Large fact-like dataset
large_df = spark.range(0, 1_000_000) \
    .withColumnRenamed("id", "txn_id") \
    .withColumn("customer_id", (F.col("txn_id") % 10_000).cast("int")) \
    .withColumn("product_id", (F.col("txn_id") % 100).cast("int")) \
    .withColumn("amount", (F.rand(seed=42) * 1000).cast("double"))

# Small dimension-like lookup
small_lookup_df = spark.range(0, 100) \
    .withColumnRenamed("id", "product_id") \
    .withColumn("category", F.concat(F.lit("Category_"), F.col("product_id")))

display(large_df.limit(5))
display(small_lookup_df.limit(5))

## Issue we are trying to solve

### Shuffle

In [0]:
agg_df = large_df.groupBy("customer_id").agg(
    F.sum("amount").alias("total_amount"),
    F.count("*").alias("txn_count")
)

display(agg_df.limit(10))
agg_df.explain(True)  # Look for Exchange and HashAggregate/SortMerge nodes

### Data Skew

In [0]:
skewed_df = spark.range(0, 1_000_000) \
    .withColumn("join_key", F.when(F.col("id") < 900_000, F.lit(1)).otherwise((F.col("id") % 100).cast("int"))) \
    .withColumn("value", F.rand(seed=7))

small_skew_lookup = spark.range(0, 100) \
    .withColumnRenamed("id", "join_key") \
    .withColumn("desc", F.concat(F.lit("Key_"), F.col("join_key")))

# Show skew
display(skewed_df.groupBy("join_key").count().orderBy(F.col("count").desc()).limit(10))

# Join (likely to produce straggler tasks)
skew_join_df = skewed_df.join(small_skew_lookup, on="join_key", how="inner")
skew_join_df.explain(True)  # Wide shuffle + potential skew

### Small Files

In [0]:
small_files_path = "/Volumes/workspace/pyspark_training/raw_data/delta_tables/"
dbutils.fs.rm(small_files_path, True)

# Force many small files
large_df.repartition(100).write.mode("overwrite").format("delta").save(small_files_path)

files = dbutils.fs.ls(small_files_path)
print("Number of output objects:", len(files))
files[:10]

## Optimization Techniques

### Boardcast join

In [0]:
# Without broadcast (Spark may still choose broadcast under AQE; for demo set threshold low if needed)
normal_join_df = large_df.join(small_lookup_df, on="product_id", how="inner")
normal_join_df.explain(True)  # Look for SortMergeJoin or ShuffledHashJoin

# Force broadcast of the small side
broadcast_join_df = large_df.join(broadcast(small_lookup_df), on="product_id", how="inner")
broadcast_join_df.explain(True)  # Look for BroadcastHashJoin
display(broadcast_join_df.limit(5))

### Repartition vs coalesce

In [0]:
print("Original partitions:", large_df.rdd.getNumPartitions())

# Repartition (can increase or decrease; full shuffle)
repart_df = large_df.repartition(24, "customer_id")
print("After repartition:", repart_df.rdd.getNumPartitions())

# Coalesce (reduce partitions; minimal shuffle)
coal_df = repart_df.coalesce(6)
print("After coalesce:", coal_df.rdd.getNumPartitions())

### Coalesce before write (fix small files)

In [0]:
optimized_files_path = "dbfs:/tmp/opt_small_files_fixed"
dbutils.fs.rm(optimized_files_path, True)

large_df.coalesce(4).write.mode("overwrite").parquet(optimized_files_path)

opt_files = dbutils.fs.ls(optimized_files_path)
print("Number of output objects after coalesce:", len(opt_files))
opt_files[:10]

### Cache / persist reused DataFrames

In [0]:
joined_df = large_df.join(broadcast(small_lookup_df), on="product_id", how="inner")

joined_df.persist(StorageLevel.MEMORY_AND_DISK)

# First action materializes the cache
joined_df.count()

# Reuse without recompute
display(joined_df.groupBy("category").agg(F.count("*").alias("cnt")).orderBy("category"))
display(joined_df.filter(F.col("amount") > 500).limit(5))

joined_df.unpersist()

### Salting to mitigate skew

In [0]:
# Add salt on the hot key (1), spread others into 0
salted_left = skewed_df.withColumn(
    "salt",
    F.when(F.col("join_key") == 1, (F.rand(seed=99) * 10).cast("int")).otherwise(F.lit(0))
)

salt_values = spark.range(0, 10).withColumnRenamed("id", "salt")
skew_key_rows = small_skew_lookup.filter(F.col("join_key") == 1).crossJoin(salt_values)
other_rows = small_skew_lookup.filter(F.col("join_key") != 1).withColumn("salt", F.lit(0))

salted_right = skew_key_rows.unionByName(other_rows)

salted_join = salted_left.join(salted_right, on=["join_key", "salt"], how="inner")
salted_join.explain(True)
display(salted_join.limit(5))